In [ ]:
import numpy as np
from scipy.spatial.distance import squareform
np.set_printoptions(linewidth=99999)
def feat(txt, m=256, b=256):
    txt = [np.sort(list(set(i)))for i in txt]
    char = (_:=np.unique(np.concat(txt),return_counts=1))[0][abs(_[1]-len(txt)//2).argpartition(m)[:m]]
    chars = np.stack(list(map(char.__getitem__,np.triu_indices(len(char),1))),1)
    res = np.stack([np.packbits(i[i.searchsorted(chars,'right')-1]==chars,1,'little')[:,0]for i in txt],1)
    cnt = np.zeros((len(chars),4),'H')
    np.add.at(cnt,(np.arange(len(chars),dtype='H')[:,None],res),1)
    ent = squareform((np.log(cnt+np.float32(1e-9))[:,None]@cnt[:,:,None])[:,0,0])[None]
    chars = char[None]
    while len(chars[0])>1:
        i,j = np.unravel_index((ent.sum((1,2))[:,None]-ent.sum(2)*2).argsort(None)[:b],ent.shape[:2])
        chars = np.stack([np.concat([chars[i][:j],chars[i][j+1:]])for i,j in zip(i,j)])
        chars,k = np.unique(chars.view(('V',chars[0].nbytes)),1)
        chars = chars[:,None].view(char.dtype)
        i = i[k]
        j = j[k]
        ent = np.stack([np.concat([np.concat([ent[i,:j,:j],ent[i,:j,j+1:]],1),np.concat([ent[i,j+1:,:j],ent[i,j+1:,j+1:]],1)])for i,j in zip(i,j)])
        print(len(chars[0]), chars[0], ent[0].sum()/len(char)-np.log(len(char)))

In [ ]:
from os import listdir
from re import findall
feat([findall(r'[a-zA-Z_][a-zA-Z0-9_]+',open('script/'+i).read())for i in listdir('script')if i[1:-4].isdigit()])

In [ ]:
from sqlite3 import connect
feat(sum(connect('cards.cdb').execute('select name from datas join texts on datas.id=texts.id where ~alias'),()))

In [ ]:
feat(sum(connect('cards.cdb').execute('select desc from datas join texts on datas.id=texts.id where ~alias'),()))